In [ ]:
# updated version!!! 01/08/2026
# Assumptions now:
# - Metadata key column is ALWAYS named: "filename"
# - "filename" values are the FULL mzML filenames (include ".mzML")
# - Join is done on basename (paths stripped) to be safe

from pathlib import Path
import pandas as pd

ROOT = Path("/nfs/turbo/lsa-gdick2/geomicro/home/sheffera/ms2_project")

# ---- EDIT THESE ----
MATCHES_STEM = "cyanometdb_matches_2026-01-07_17-45-21"
REF_STEM     = "AB_indiv_summary_perfile_with_ms1_auc_2026-01-07_17-44-31"
META_PATH    = Path("/nfs/turbo/lsa-gdick2/geomicro/home/sheffera/ms2_project/cpcc_metadata-01-08-2026.csv")

OUTPUT_NAME  = "test_AB_matches_by_file_with_auc_and_metadata.csv"


# -------------------- file finding --------------------
def find_best_table_file(root: Path, stem: str) -> Path:
    hits = list(root.rglob(f"{stem}*"))
    if not hits:
        raise FileNotFoundError(f"Could not find any file starting with '{stem}' under {root}")

    pref = {".csv": 0, ".tsv": 1, ".txt": 2, ".xlsx": 3, ".xls": 4}
    hits.sort(key=lambda p: (pref.get(p.suffix.lower(), 99), -p.stat().st_mtime))
    return hits[0]


# -------------------- robust reading --------------------
def read_table(path: Path) -> pd.DataFrame:
    ext = path.suffix.lower()
    if ext in [".xlsx", ".xls"]:
        df = pd.read_excel(path, dtype=str, engine="openpyxl")
        print(f"[INFO] Loaded EXCEL {path.name} | rows={len(df):,} cols={df.shape[1]}")
        return df

    encodings = ["utf-8", "latin-1"]
    seps = ["\t", ",", ";"]

    last_err = None
    for enc in encodings:
        for sep in seps:
            try:
                df = pd.read_csv(path, sep=sep, dtype=str, encoding=enc, engine="python")
                if df.shape[1] > 1:
                    print(f"[INFO] Loaded TEXT {path.name} | rows={len(df):,} cols={df.shape[1]} sep={repr(sep)} enc={enc}")
                    return df
            except Exception as e:
                last_err = e
    raise RuntimeError(f"Failed to read {path}. Last error: {last_err}")


# -------------------- helpers --------------------
def clean_list_cell(x):
    """Parse list-like cells: '1,2,3' or '[1, 2, 3]' or "'1,2,3'" """
    if pd.isna(x):
        return []
    s = str(x).strip()
    if not s:
        return []
    s = s.replace("[", "").replace("]", "").replace("(", "").replace(")", "")
    s = s.replace("'", "").replace('"', "")
    s = s.strip()
    if not s:
        return []
    return [v.strip() for v in s.split(",") if v.strip()]

def basename_str(s: str) -> str:
    s = str(s).replace("\\", "/")
    return s.split("/")[-1].strip()

def canonical_scan_ids(scan_ids_cell) -> str:
    """Stable ID for a pooled feature: sorted unique ints joined by commas."""
    lst = clean_list_cell(scan_ids_cell)
    nums = []
    for v in lst:
        try:
            nums.append(int(float(v)))
        except Exception:
            continue
    nums = sorted(set(nums))
    return ",".join(str(n) for n in nums)

def normalize_filename(x) -> str:
    """Basename-only normalization for safe joining across paths."""
    if pd.isna(x):
        return ""
    return basename_str(str(x))


# -------------------- main --------------------
matches_path = find_best_table_file(ROOT, MATCHES_STEM)
ref_path     = find_best_table_file(ROOT, REF_STEM)

print("[INFO] Using matches:", matches_path)
print("[INFO] Using ref:    ", ref_path)

matches = read_table(matches_path)
ref     = read_table(ref_path)

# ---- sanity requirements ----
for col in ["merged_precmz", "scan_ids", "files"]:
    if col not in matches.columns:
        raise KeyError(f"Matches table missing required column: {col}")

for col in ["scan_ids", "source_file", "ms1_auc"]:
    if col not in ref.columns:
        raise KeyError(f"Reference table missing required column: {col}")

# ---------------------------
# 1) Build a feature_key in BOTH tables (from scan_ids)
# ---------------------------
m = matches.copy()
r = ref.copy()

m["feature_key"] = m["scan_ids"].apply(canonical_scan_ids)
r["feature_key"] = r["scan_ids"].apply(canonical_scan_ids)

# ---------------------------
# 2) Deconvolute matches by FILE: explode pooled "files" into one row per file
# ---------------------------
m["_files_list"] = m["files"].apply(clean_list_cell).apply(lambda lst: [normalize_filename(x) for x in lst])
m = m.explode("_files_list", ignore_index=True)
m = m.rename(columns={"_files_list": "file"})
m["file"] = m["file"].astype(str).str.strip()
m = m[m["file"].ne("")].copy()

print(f"[INFO] Matches exploded by file -> {len(m):,} rows (one per feature_key × file)")

# ---------------------------
# 3) Prep reference: normalize source_file basename into ref_file
# ---------------------------
r["ref_file"] = r["source_file"].apply(normalize_filename)

# numeric conversions (optional but helpful)
for col in ["merged_precmz", "ms1_auc", "ref_ms1_auc", "ms1_auc_over_ref", "charge", "rt_min", "rt_median", "rt_max"]:
    if col in m.columns:
        m[col] = pd.to_numeric(m[col], errors="coerce")
    if col in r.columns:
        r[col] = pd.to_numeric(r[col], errors="coerce")

# ---------------------------
# 4) Merge: feature_key + file -> AUC from that file
# ---------------------------
out = m.merge(
    r,
    left_on=["feature_key", "file"],
    right_on=["feature_key", "ref_file"],
    how="left",
    suffixes=("", "_ref")
)

missing_auc = out["ms1_auc"].isna().sum() if "ms1_auc" in out.columns else -1
print(f"[INFO] Rows missing ms1_auc after merge: {missing_auc:,} / {len(out):,}")

# ---------------------------
# 5) Add metadata using FULL filename (.mzML included)
#    Metadata key column must be exactly: "filename"
# ---------------------------
meta_df = read_table(META_PATH).copy()

if "filename" not in meta_df.columns:
    raise KeyError(
        "Metadata table must include a column named exactly 'filename' "
        "(full mzML filename, including '.mzML'). "
        f"Found columns: {list(meta_df.columns)}"
    )

meta_df["filename_key"] = meta_df["filename"].apply(normalize_filename)
out["filename_key"] = out["file"].apply(normalize_filename)

# (Optional) If you ever get mismatches because metadata omits extension, uncomment this fallback:
# meta_df["filename_key"] = meta_df["filename_key"].str.replace(r"\.mzML$", "", regex=True)
# out["filename_key"] = out["filename_key"].str.replace(r"\.mzML$", "", regex=True)

out2 = out.merge(
    meta_df,
    on="filename_key",
    how="left",
    suffixes=("", "_meta")
)

# keep a clean filename column in output (requested) that matches the mzML file basename
out2["filename"] = out2["file"]

# quick missing-metadata diagnostic
meta_payload_cols = [c for c in meta_df.columns if c not in ["filename_key"]]
missing_meta_rows = out2[meta_payload_cols].isna().all(axis=1).sum() if meta_payload_cols else 0
print(f"[INFO] Rows missing metadata after merge: {missing_meta_rows:,} / {len(out2):,}")

# ---------------------------
# 6) Reorder key columns to front (robust: include what exists)
# ---------------------------
front = [
    "merged_precmz",
    "charge",
    "rt_min", "rt_median", "rt_max",
    "feature_key",
    "filename",     
    "ms1_auc", "ref_ms1_auc", "ms1_auc_over_ref",
]
front = [c for c in front if c in out2.columns]
rest  = [c for c in out2.columns if c not in front and c != "file"]  # drop 'file' from output by default
out2 = out2[front + rest]

# ---------------------------
# 7) Save
# ---------------------------
out_path = matches_path.parent / OUTPUT_NAME
out2.to_csv(out_path, index=False)
print("[DONE] Wrote:", out_path)